In [9]:
import os
import cv2
import numpy as np

gestures = ["palm", "fist", "ok", "peace", "thumbs_up"]

base_dir = "gesture_dataset"

for gesture in gestures:
    path = os.path.join(base_dir, gesture)
    os.makedirs(path, exist_ok=True)

In [10]:
for gesture in gestures:
    for i in range(200):  # 200 images per class → 1000 total

        img = np.zeros((224,224,3), dtype=np.uint8)

        # Draw different shapes for each gesture
        if gesture == "palm":
            cv2.rectangle(img, (50,50), (170,170), (255,255,255), -1)

        elif gesture == "fist":
            cv2.circle(img, (112,112), 60, (255,255,255), -1)

        elif gesture == "ok":
            cv2.circle(img, (112,112), 40, (255,255,255), 5)

        elif gesture == "peace":
            cv2.line(img, (80,50), (80,170), (255,255,255), 10)
            cv2.line(img, (140,50), (140,170), (255,255,255), 10)

        elif gesture == "thumbs_up":
            cv2.arrowedLine(img, (112,170), (112,50), (255,255,255), 10)

        cv2.imwrite(f"{base_dir}/{gesture}/{gesture}_{i}.png", img)

print("✅ Synthetic dataset created")

✅ Synthetic dataset created


In [11]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = train_datagen.flow_from_directory(
    "gesture_dataset",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    "gesture_dataset",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 800 images belonging to 5 classes.
Found 200 images belonging to 5 classes.


In [12]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_data, validation_data=val_data, epochs=5)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 79s 3s/step - accuracy: 0.9488 - loss: 0.0953 - val_accuracy: 1.0000 - val_loss: 6.4849e-06
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 74s 3s/step - accuracy: 1.0000 - loss: 6.2331e-07 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 75s 3s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 75s 3s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 74s 3s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00


In [14]:
gesture_meaning = {
    "palm": "Stop",
    "fist": "Select",
    "ok": "Confirm",
    "peace": "Victory",
    "thumbs_up": "Like"
}

def gesture_summary(gesture):
    print("\n✋ Gesture Report")
    print("="*30)
    print("Gesture:", gesture)
    print("Action :", gesture_meaning.get(gesture, "Unknown"))

“Due to dataset constraints, a synthetic dataset was generated to simulate gesture patterns, and the model was trained to validate the pipeline.”